## Importing Required Libraries

We first import the necessary libraries, including:
- `folium` for interactive map visualization 
- `pandas` for data manipulation
- `sqlalchemy` and `sqlite3` for handling databases
- `re` and `unicodedata` for text processing

# NB3B - Map

This notebook processes Pokémon-related data from SQL databases and generates a Pokémon-themed map using Folium.

In [1]:
import folium
import sqlalchemy 
import pandas as pd
import os
import sqlite3
import re
import unicodedata
import random

from folium import FeatureGroup
from folium.plugins import HeatMap, MarkerCluster, Search
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy import create_engine, Column, Integer, String, Enum, Float
from folium import IFrame
from branca.colormap import linear
from IPython.display import HTML



from branca.element import MacroElement
from jinja2 import Template

## Loading and Merging Pokémon Data with Environmental Locations  

To analyze the relationship between Pokémon types and environmental conditions, we need to retrieve data from a SQLite database (**`main.db`**). This includes:  

- **Fire-type Pokémon** matched with the **hottest places**  
- **Water-type Pokémon** matched with the **wettest places**  
- **Ice-type Pokémon** matched with the **coldest places**  

We achieve this by performing **SQL joins** between Pokémon tables and corresponding climate tables according to their rankings.

In [2]:
conn = sqlite3.connect('../../data/main.db')

# Merging the tables in the database
fire_query = '''
SELECT f.*, h.* 
  FROM fire_pokemon f
  JOIN hottest_places h
    ON f.ranking = h.ranking
'''

water_query = ''' 
SELECT w1.*, w2.*
  FROM water_pokemon w1
  JOIN wettest_places w2
    ON w1.ranking = w2.ranking
'''

ice_query = ''' 
SELECT i.*, c.*
  FROM ice_pokemon i
  JOIN coldest_places c
    ON i.ranking = c.ranking
'''

# Load the data into separate pandas DataFrames
fire_df = pd.read_sql_query(fire_query, conn)
water_df = pd.read_sql_query(water_query, conn)
ice_df = pd.read_sql_query(ice_query, conn)

# Drop duplicate 'ranking' column (keeping only one)
fire_df = fire_df.loc[:, ~fire_df.columns.duplicated()]
water_df = water_df.loc[:, ~water_df.columns.duplicated()]
ice_df = ice_df.loc[:, ~ice_df.columns.duplicated()]

## Cleaning and Fixing Pokémon Descriptions  

Pokémon descriptions may contain various formatting issues, such as:  
- Incorrect capitalization (e.g., `POKéMON` → `Pokémon`)  
- Extra spaces or special characters  
- Incorrect possessive forms (e.g., `Trainers` → `Trainer's`)  
- Missing punctuation (e.g., `doesnt` → `doesn't`)  

A custom function is applied to automatically fix these errors, ensuring that all descriptions are properly formatted before they are used in the final dataset.


In [3]:
# Define the grammar fix function
def fix_grammar_in_dataframe(df):
    # Function to fix the grammar of a single Pokémon description
    def fix_grammar(description):
        # Fix "POKéMON" to "Pokémon"
        description = re.sub(r'POKéMON', 'Pokémon', description)
        # Normalize Unicode characters (e.g., handle special characters)
        description = unicodedata.normalize("NFKC", description)
        # Replace double spaces with a single space
        description = re.sub(r'\s{2,}', ' ', description)
        # Fix broken word "con­tinuously" (which might have hidden characters)
        description = re.sub(r'con­tinuously', 'continuously', description)
        description = re.sub(r'swim ming', 'swimming', description)
        # Merge words with a hyphen, such as "X- Y" to "X-Y"
        description = re.sub(r'(\w)- (\w)', r'\1-\2', description)
        # Capitalize all uppercase words
        description = re.sub(r'\b([A-Z]+)\b', lambda match: match.group(0).capitalize(), description)
        # Ensure a space between words if there is a form feed character
        description = re.sub(r'(\w)\f(\w)', r'\1 \2', description)
        # Add space after full stops if missing
        description = re.sub(r'\.(\S)', r'. \1', description)
        # Ensure space after full stop and before form feed character
        description = re.sub(r'\.(\f)', r'. \1', description)
        # Remove unwanted characters (non-alphanumeric, non-whitespace)
        description = re.sub(r'[^\w\s.,;?!\'"-]', '', description)
        # Fix broken word "Un able" to "Unable"
        description = re.sub(r'\bUn\s+able\b', 'Unable', description)
        # Remove non-printable characters
        description = ''.join(char for char in description if char.isprintable())
        # Fix "Its" followed by certain words to "It's"
        description = re.sub(r'\bIts\b(\s+(highly|feeling|apparently|said|not))', r"It's\1", description)
        # Keep "Its" if followed by a noun (indicating possession)
        description = re.sub(r'\bIts\b(\s+[a-zA-Z]+)', lambda m: m.group(0), description)
        # Fix common mistakes like "doesnt" to "doesn't" and "cant" to "can't"
        description = re.sub(r"\bdoesnt\b", "doesn't", description)
        description = re.sub(r"\bcant\b", "can't", description)
        # Standardize "UFO" to uppercase
        description = re.sub(r'\bufo\b', "UFO", description, flags=re.IGNORECASE)
        
        # Ensure possessive "Trainer's" is used correctly
        if "Trainer's" not in description:
            description = re.sub(r'\bTrainers\b', "Trainer's", description)
        
        # Ensure possessive "Pokémon's" is used correctly
        if "Pokémon's" not in description:
            description = re.sub(r'\bPokémons\b', "Pokémon's", description)
        
        # Fix possessive "Skeledirges" to "Skeledirge's"
        description = re.sub(r'\bSkeledirges\b', "Skeledirge's", description)
        # Fix possessive "Barraskewdas" to "Barraskewda's"
        description = re.sub(r'\Barraskewdas\b', "Barraskewda's", description)
        # Fix possessive "Regices" to "Regice's"
        description = re.sub(r'\bRegices\b', "Regice's", description)
        # Fix "Pokémonanything" to "Pokémon - anything"
        description = re.sub(r'Pokémonanything', 'Pokémon - anything', description)
        # Fix "gather ing" to "gathering"
        description = re.sub(r' gather ing', ' gathering', description)
        
        return description

    # Apply the grammar fix function to the "pokemon_description" column
    df['pokemon_description'] = df['pokemon_description'].apply(fix_grammar)
    
    return df

# Apply the function to clean the descriptions
ice_df = fix_grammar_in_dataframe(ice_df)
fire_df = fix_grammar_in_dataframe(fire_df)
water_df = fix_grammar_in_dataframe(water_df)

## Visualizing Pokémon Distribution on the Map  

After merging Pokémon data with geographical and climate information, we can create a detailed map using **Folium** that displays Pokémon locations across different regions.  

### **Map Features:**  
1. **Pokémon Icons for Locations** – Each Pokémon icon represents a specific Pokémon's observed location and shows their basic stats and description, making it easy to identify their distribution.  
2. **Integrated Heatmap with Climate Data** – The map includes a heatmap layer that visualizes temperature or rainfall intensity across regions, with markers at different regions that can be clicked on to display information (names of regions, rainfall/ temperature).  
3. **Temperature and Rainfall Scale** – A scale is provided to interpret climate variations, helping to correlate Pokémon distribution with environmental conditions.  
4. **Toggle Layers for Custom Views** – Users can switch between different layers, choosing to display different types of Pokémons (water, fire, ice) and a heatmap of different regions (temperature, rainfall) or a combination of these.  

### **Key Insights from the Map:**  
- The distribution of Water-type Pokémon aligns with areas experiencing high rainfall.  
- Fire-type Pokémon tend to appear in warmer regions, while Ice-type Pokémon are more common in colder climates.  
- Comparing Pokémon distribution with environmental data can help identify trends in their habitat preferences. 

This visualization provides a dynamic way to analyze Pokémon distribution in relation to environmental factors, enhancing our understanding of their habitats.  


In [4]:
# Initialize map
map_center = [20.0, 0.0]  # World center
pokemon_map = folium.Map(tiles = None, location=map_center, zoom_start=2)
folium.TileLayer('openstreetmap').add_to(pokemon_map)


# Create MarkerCluster groups for Fire, Ice, and Water Pokémon
fire_cluster = MarkerCluster(name="Fire Pokémon").add_to(pokemon_map)
ice_cluster = MarkerCluster(name="Ice Pokémon").add_to(pokemon_map)
water_cluster = MarkerCluster(name="Water Pokémon").add_to(pokemon_map)

# Dictionary to store previously placed marker locations
placed_locations = {}

#List to store Pokemon locations for search bar
pokemon_features = []

def add_pokemon_markers(pokemon_df, cluster_group):
    for index, pokemon in pokemon_df.iterrows():  # Iterate over DataFrame rows
        lat, lon = pokemon["latitude"], pokemon["longitude"]

        # Ensure values exist and handle missing data gracefully
        name = pokemon.get("name", "Unknown")
        description = pokemon.get("pokemon_description", "No description available")
        stats = pokemon.get("total_stat", "N/A")
        ranking = pokemon.get("ranking", "Unranked")
        portrait_url = pokemon.get("pokemon_portrait", "https://via.placeholder.com/150")

        # Fetch full stats (handling missing values)
        hp = pokemon.get("hp_stat", "N/A")
        attack = pokemon.get("attack_stat", "N/A")
        defense = pokemon.get("defense_stat", "N/A")
        sp_attack = pokemon.get("special_attack_stat", "N/A")
        sp_defense = pokemon.get("special_defense_stat", "N/A")
        speed = pokemon.get("speed_stat", "N/A")

        # Create popup HTML for Pokémon details
        popup_html = f"""
        <div style="text-align: center; width: 240px; max-width: 240px; padding: 10px; background-color: white; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); word-wrap: break-word; overflow-wrap: break-word;">
            <h4 style="font-size: 16px; font-weight: bold; color: #D34B29; margin: 0;">{name}</h4>
            <img src="{portrait_url}" width="150px" style="cursor: pointer; border-radius: 8px; max-width: 100%; height: auto;">
            <br><br>
            <b style="font-size: 14px;">Ranking:</b> <span style="font-size: 12px; color: #555;">{ranking}</span><br>
            <b style="font-size: 14px;">Total Stats:</b> <span style="font-size: 12px; color: #555;">{stats}</span><br>
            <hr>
            <b style="font-size: 14px;">HP:</b> <span style="font-size: 12px; color: #555;">{hp}</span><br>
            <b style="font-size: 14px;">Attack:</b> <span style="font-size: 12px; color: #555;">{attack}</span><br>
            <b style="font-size: 14px;">Defense:</b> <span style="font-size: 12px; color: #555;">{defense}</span><br>
            <b style="font-size: 14px;">Sp. Attack:</b> <span style="font-size: 12px; color: #555;">{sp_attack}</span><br>
            <b style="font-size: 14px;">Sp. Defense:</b> <span style="font-size: 12px; color: #555;">{sp_defense}</span><br>
            <b style="font-size: 14px;">Speed:</b> <span style="font-size: 12px; color: #555;">{speed}</span><br>
            <hr>
            <p style="font-size: 12px; text-align: justify; color: #333; margin-top: 8px; line-height: 1.5; word-wrap: break-word; overflow-wrap: break-word; white-space: normal; max-width: 200px; padding: 0; margin: 0;">{description}</p>
        </div>
        """
        # Create a popup instead of a tooltip for better interaction
        popup = folium.Popup(popup_html, max_width=300)

        # Create a DivIcon for Pokémon icon
        icon_html = f"""
        <div style="background: url('{portrait_url}') no-repeat center center; background-size: contain; width: 50px; height: 50px;"></div>
        """
        div_icon = folium.DivIcon(html=icon_html)

        # Check if location is already taken
        key = (round(lat, 3), round(lon, 3))  # Rounded to 3 decimal places to reduce precision issues

        # Apply larger offset if location is already occupied
        if key in placed_locations:
            # Apply stagger effect for latitude and longitude with larger offset values
            offset_lat = lat + 0.002 * (placed_locations[key] % 5)  # Larger offset for latitude
            offset_lon = lon + 0.002 * (placed_locations[key] // 5)  # Larger offset for longitude
            placed_locations[key] += 1  # Increment counter for that location
        else:
            offset_lat = lat
            offset_lon = lon
            placed_locations[key] = 1  # First marker at this location
        
        # Add Pokémon to the search dataset
        pokemon_features.append({
            "type": "Feature",
            "properties": {"name": name},
            "geometry": {"type": "Point", "coordinates": [offset_lon, offset_lat]},
        })
        # Create marker with popup for Pokémon
        marker = folium.Marker(
            location=[offset_lat, offset_lon],
            popup=popup,  # Popup for the information
            icon=div_icon  # Pokémon image as marker icon
        )

        # Add marker to the cluster group
        marker.add_to(cluster_group)
        
        
# Add Pokémon markers to clusters
add_pokemon_markers(fire_df, fire_cluster)
add_pokemon_markers(ice_df, ice_cluster)
add_pokemon_markers(water_df, water_cluster)

# Add clusters to the map
pokemon_map.add_child(fire_cluster)
pokemon_map.add_child(ice_cluster)
pokemon_map.add_child(water_cluster)


In [5]:
# Create a set to store the positions of already added markers
added_positions = set()

# Create color scales using min/max temperature and rainfall values from dataframes
colormaps = {
    "temp": linear.YlOrRd_09.scale(ice_df["temperature"].min(), fire_df["temperature"].max()),
    "rain": linear.Blues_09.scale(water_df["max_rainfall"].min(), water_df["max_rainfall"].max())
}

# Function to add temperature/rainfall circle markers with hover tooltips and offset only when overlapping
def add_circle_marker(row, metric, colormap, layer, is_pokemon=False):
    unit = "°C" if metric == "temperature" else "mm"  # Assign correct units
    popup_content = f"Region: {row.region}<br>{metric.replace('_', ' ').capitalize()}: {row[metric]} {unit}"
    color = colormap(row[metric])

    # Increase marker radius for better visibility
    radius = 10 if not is_pokemon else 5  # Larger radius for temperature/rainfall markers

    # Default latitude and longitude
    lat, lon = row.latitude, row.longitude
    
    # Apply random offset if the position is already occupied
    offset_lat, offset_lon = lat, lon
    offset_found = False
    attempts = 0

    while (offset_lat, offset_lon) in added_positions and attempts < 10:  # Limit attempts to avoid infinite loop
        # Apply a random offset to the position
        offset_lat = lat + random.uniform(-0.002, 0.002)  # Random latitude offset
        offset_lon = lon + random.uniform(-0.002, 0.002)  # Random longitude offset
        attempts += 1
        offset_found = True

    # Add the position to the set of added positions
    added_positions.add((offset_lat, offset_lon))

    # Create the marker with the (possibly offset) position
    if is_pokemon:
        # Pokémon marker click functionality (show info on click)
        marker = folium.Marker(
            location=[offset_lat, offset_lon],
            popup=folium.Popup(popup_content, max_width=300),
            icon=folium.Icon(color='blue', icon='info-sign')  # Icon for Pokémon
        )
    else:
        # Temperature/Rainfall marker (show info on hover)
        marker = folium.CircleMarker(
            location=[offset_lat, offset_lon],
            radius=radius,  # Increased radius
            popup=folium.Popup(popup_content, max_width=300),
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.8,
            z_index_offset=1000  # Higher z-index than Pokémon markers
        )
        tooltip = folium.Tooltip(popup_content)  # Correct way to add tooltip for hover
        marker.add_child(tooltip)

    marker.add_to(layer)

# Function to add heatmap markers without offsets (heatmap stays in the original position)
def add_heatmap_markers(df, col_name, colormap, heatmap_layer):
    heat_data = []
    for _, row in df.iterrows():
        lat, lon = row.latitude, row.longitude
        heat_data.append([lat, lon, row.get(col_name)])

    HeatMap(heat_data, min_opacity=0.2, max_opacity=0.8, radius=15, blur=20, gradient=colormap).add_to(heatmap_layer)


# Create heatmap layer
heatmap_layer = folium.FeatureGroup(name="HeatMap")

# Apply function to hottest & coldest places (temperature-based)
fire_df.apply(add_circle_marker, axis=1, args=("temperature", colormaps["temp"], heatmap_layer))
ice_df.apply(add_circle_marker, axis=1, args=("temperature", colormaps["temp"], heatmap_layer))

# Apply function to wettest places (rainfall-based)
water_df.apply(add_circle_marker, axis=1, args=("max_rainfall", colormaps["rain"], heatmap_layer))

# Generate heatmap data for temperature (without random offset)
heat_data_temp = [
    [row.latitude,  # Latitude stays the same for heatmap
     row.longitude,  # Longitude stays the same for heatmap
     row.temperature] 
    for _, row in pd.concat([fire_df, ice_df]).iterrows()
]

# Generate heatmap data for rainfall (without random offset)
heat_data_rain = [
    [row.latitude,  # Latitude stays the same for heatmap
     row.longitude,  # Longitude stays the same for heatmap
     row.max_rainfall] 
    for _, row in water_df.iterrows()
]

# Add HeatMap layers
HeatMap(heat_data_temp).add_to(heatmap_layer)
HeatMap(heat_data_rain).add_to(heatmap_layer)

# Add legends and layers to map
colormaps["temp"].caption = 'Temperature Scale (°C)'
colormaps["rain"].caption = 'Rainfall Scale (mm)'
colormaps["temp"].add_to(pokemon_map)
colormaps["rain"].add_to(pokemon_map)

# Add heatmap to the pokemon map
pokemon_map.add_child(heatmap_layer)




## Adding Search Bar to Map

We create a search bar to identify Pokémon by location.

### **Search Bar Features:**  
1. **List of Pokémon** – The search bar shows a list of Pokémon that is accessible.
2. **Map Indicator** – The map shows a red spot to indicate the Pokémon that has been searched.



### **Notes:**
Due to the way the Search plugin works in Folium, we cannot deselect the list of all Pokemon locations nor make their markers invisible upon initialisation. This has to be manually done upon loading into the map.




In [6]:
geojson_data = {
    "type": "FeatureCollection",
    "features": []
}

# Transform the original data into GeoJson data format
for item in pokemon_features:
    feature = {
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": item['geometry']['coordinates']
        },
        "properties": {
            "name": item['properties']['name']
        }
    }
    geojson_data["features"].append(feature)


# Add GeoJSON layer 
geojson_layer = folium.GeoJson(geojson_data, name="Pokemon Locations", show=False).add_to(pokemon_map)

#Add search bar
search = Search(
    layer=geojson_layer,
    search_label="name",  # Correct search label, "name" is the key in properties
    geom_type="Point",  # We're dealing with point locations
    placeholder="Search Pokémon...",  # Text in the search box
    collapsed=False,  # Keeps the search bar expanded
    zoom_to_feature=True,  # Zoom in to the feature when clicked
    zoom = 10
).add_to(pokemon_map)

pokemon_map



In [7]:
folium.LayerControl().add_to(pokemon_map)

# Specify the directory where the map will be saved
save_directory = "../../visuals/html_files/"

# Create the directory if it doesn't exist
os.makedirs(save_directory, exist_ok=True)

# Save the map to the specified directory
pokemon_map.save(f"{save_directory}pokemon_map.html")

print(f"Map generated: {save_directory}pokemon_map.html")

Map generated: ../../visuals/html_files/pokemon_map.html
